In [1]:
import warnings
from itertools import product

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [4]:
def fallback_prepare_data_from_raw(train_path="train.csv", test_path="test.csv"):
    train_raw = pd.read_csv(train_path)
    test_raw = pd.read_csv(test_path)

    y_local = train_raw["Transported"].astype(int).copy()

    combined = pd.concat([train_raw.drop(columns=["Transported"]), test_raw], axis=0, ignore_index=True)

    combined["CryoSleep"] = combined["CryoSleep"].astype("boolean").fillna(False).astype(int)
    combined["VIP"] = combined["VIP"].astype("boolean").fillna(False).astype(int)

    if "Cabin" in combined.columns:
        cabin_parts = combined["Cabin"].fillna("Unknown/0/Unknown").str.split("/", expand=True)
        combined["Deck"] = cabin_parts[0]
        combined["CabinNum"] = pd.to_numeric(cabin_parts[1], errors="coerce")
        combined["Side"] = cabin_parts[2]

    drop_cols = ["Name", "Cabin"]
    combined = combined.drop(columns=[c for c in drop_cols if c in combined.columns], errors="ignore")

    for col in combined.columns:
        if combined[col].dtype == "object" and col != "PassengerId":
            combined[col] = combined[col].fillna("Unknown")

    num_cols = combined.select_dtypes(include=["number"]).columns.tolist()
    for col in num_cols:
        combined[col] = combined[col].fillna(combined[col].median())

    cat_cols = combined.select_dtypes(include=["object"]).columns.tolist()
    if "PassengerId" in cat_cols:
        cat_cols.remove("PassengerId")

    combined = pd.get_dummies(combined, columns=cat_cols, drop_first=True)

    train_part = combined.iloc[: len(train_raw)].copy()
    test_part = combined.iloc[len(train_raw):].copy()

    X_local = train_part.drop(columns=["PassengerId"], errors="ignore")
    X_test_local = test_part.drop(columns=["PassengerId"], errors="ignore")
    passenger_ids_local = test_raw["PassengerId"].copy()

    return X_local, y_local, X_test_local, passenger_ids_local


if all(v in globals() for v in ["X", "y", "X_test"]):
    print("Using provided preprocessed data: X, y, X_test")
else:
    print("X, y, X_test not found. Building fallback features from raw train/test.csv")
    X, y, X_test, passenger_ids = fallback_prepare_data_from_raw()

if not isinstance(X, pd.DataFrame) or not isinstance(X_test, pd.DataFrame):
    raise TypeError("X and X_test must be pandas DataFrame objects")

if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]
y = pd.Series(y).astype(int).reset_index(drop=True)

if X.isna().any().any() or X_test.isna().any().any():
    raise ValueError("X and X_test must not contain missing values")

non_numeric_train = X.select_dtypes(exclude=["number", "bool"]).columns.tolist()
non_numeric_test = X_test.select_dtypes(exclude=["number", "bool"]).columns.tolist()
if non_numeric_train or non_numeric_test:
    raise ValueError(
        f"All features must be numeric. Non-numeric columns found. "
        f"Train: {non_numeric_train}, Test: {non_numeric_test}"
    )

for frame in (X, X_test):
    bool_cols = frame.select_dtypes(include="bool").columns
    if len(bool_cols) > 0:
        frame[bool_cols] = frame[bool_cols].astype(int)

if "passenger_ids" not in globals():
    if "test" in globals() and isinstance(test, pd.DataFrame) and "PassengerId" in test.columns:
        passenger_ids = test["PassengerId"].copy()
    else:
        passenger_ids = pd.read_csv("test.csv")["PassengerId"]

print("Data checks passed")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

X, y, X_test not found. Building fallback features from raw train/test.csv
Data checks passed
X shape: (8693, 25)
y shape: (8693,)
X_test shape: (4277, 25)


In [5]:
THRESHOLDS = np.round(np.arange(0.48, 0.551, 0.01), 3)
N_SPLITS = 5

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

models = {
    "lgbm_main": LGBMClassifier(
        n_estimators=900,
        learning_rate=0.02,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=25,
        reg_alpha=0.1,
        reg_lambda=0.3,
        random_state=RANDOM_STATE,
        verbosity=-1,
    ),
    "lgbm_alt": LGBMClassifier(
        n_estimators=700,
        learning_rate=0.035,
        max_depth=6,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.75,
        min_child_samples=35,
        reg_alpha=0.0,
        reg_lambda=0.6,
        random_state=RANDOM_STATE + 7,
        verbosity=-1,
    ),
    "xgboost": XGBClassifier(
        n_estimators=900,
        learning_rate=0.025,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        use_label_encoder=False,
        tree_method="hist",
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=1200,
        max_depth=14,
        min_samples_split=8,
        min_samples_leaf=3,
        max_features="sqrt",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

print("Models configured:", list(models.keys()))
print("Threshold grid:", THRESHOLDS.tolist())

Models configured: ['lgbm_main', 'lgbm_alt', 'xgboost', 'random_forest']
Threshold grid: [0.48, 0.49, 0.5, 0.51, 0.52, 0.53, 0.54, 0.55]


In [6]:
def best_threshold_score(y_true, probas, thresholds):
    best_thr = None
    best_score = -1.0
    for thr in thresholds:
        preds = (probas >= thr).astype(int)
        score = accuracy_score(y_true, preds)
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr, float(best_score)


def cv_oof_predictions(model, X, y, skf):
    oof = np.zeros(len(X), dtype=float)
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        clf = clone(model)
        clf.fit(X_tr, y_tr)
        va_proba = clf.predict_proba(X_va)[:, 1]
        oof[va_idx] = va_proba

        fold_thr, fold_score = best_threshold_score(y_va.values, va_proba, THRESHOLDS)
        fold_scores.append(fold_score)
        print(f"  Fold {fold}: acc={fold_score:.5f} at thr={fold_thr:.2f}")

    best_thr, best_cv = best_threshold_score(y.values, oof, THRESHOLDS)
    return {
        "oof": oof,
        "fold_scores": fold_scores,
        "best_threshold": best_thr,
        "best_cv": best_cv,
    }

In [16]:
def search_weighted_ensemble(oof_pred_map, y, thresholds, random_trials=8000):
    model_names = list(oof_pred_map.keys())
    pred_matrix = np.column_stack([oof_pred_map[name] for name in model_names])

    best = {
        "weights": None,
        "threshold": None,
        "score": -1.0,
    }

    step = 0.05
    grid = np.arange(0, 1 + 1e-9, step)

    for w1 in grid:
        for w2 in grid:
            for w3 in grid:
                w4 = 1.0 - (w1 + w2 + w3)
                if w4 < -1e-9:
                    continue
                w4 = max(0.0, w4)
                weights = np.array([w1, w2, w3, w4], dtype=float)
                if np.abs(weights.sum() - 1.0) > 1e-8:
                    continue

                blend = pred_matrix @ weights
                thr, score = best_threshold_score(y.values, blend, thresholds)
                if score > best["score"]:
                    best = {
                        "weights": weights.copy(),
                        "threshold": thr,
                        "score": score,
                    }

    rng = np.random.default_rng(RANDOM_STATE)
    for _ in range(random_trials):
        weights = rng.dirichlet(np.ones(pred_matrix.shape[1]))
        blend = pred_matrix @ weights
        thr, score = best_threshold_score(y.values, blend, thresholds)
        if score > best["score"]:
            best = {
                "weights": weights.copy(),
                "threshold": thr,
                "score": score,
            }

    return model_names, best

In [8]:
model_cv_results = {}
oof_pred_map = {}

for model_name, model in models.items():
    print(f"\nTraining CV for {model_name}")
    result = cv_oof_predictions(model, X, y, skf)
    model_cv_results[model_name] = result
    oof_pred_map[model_name] = result["oof"]
    print(
        f"{model_name} | best CV={result['best_cv']:.5f} "
        f"at threshold={result['best_threshold']:.2f}"
    )


Training CV for lgbm_main
  Fold 1: acc=0.81771 at thr=0.55
  Fold 2: acc=0.80736 at thr=0.51
  Fold 3: acc=0.80679 at thr=0.48
  Fold 4: acc=0.81646 at thr=0.54
  Fold 5: acc=0.79402 at thr=0.51
lgbm_main | best CV=0.80686 at threshold=0.48

Training CV for lgbm_alt
  Fold 1: acc=0.81369 at thr=0.48
  Fold 2: acc=0.80966 at thr=0.54
  Fold 3: acc=0.80219 at thr=0.48
  Fold 4: acc=0.82221 at thr=0.50
  Fold 5: acc=0.79287 at thr=0.54
lgbm_alt | best CV=0.80571 at threshold=0.48

Training CV for xgboost
  Fold 1: acc=0.81714 at thr=0.52
  Fold 2: acc=0.81024 at thr=0.48
  Fold 3: acc=0.80276 at thr=0.50
  Fold 4: acc=0.82278 at thr=0.50
  Fold 5: acc=0.80150 at thr=0.53
xgboost | best CV=0.80893 at threshold=0.52

Training CV for random_forest
  Fold 1: acc=0.81369 at thr=0.49
  Fold 2: acc=0.79988 at thr=0.55
  Fold 3: acc=0.81081 at thr=0.50
  Fold 4: acc=0.81761 at thr=0.50
  Fold 5: acc=0.79517 at thr=0.48
random_forest | best CV=0.80651 at threshold=0.50


In [17]:
def stacking_oof_predictions(meta_features, y, skf, thresholds, c_grid=(0.25, 0.5, 1.0, 2.0, 4.0)):
    best_result = {
        "oof": None,
        "threshold": None,
        "score": -1.0,
        "best_c": None,
    }

    for c_value in c_grid:
        stack_oof = np.zeros(len(y), dtype=float)

        for fold, (tr_idx, va_idx) in enumerate(skf.split(meta_features, y), start=1):
            X_meta_tr, X_meta_va = meta_features[tr_idx], meta_features[va_idx]
            y_meta_tr = y.iloc[tr_idx]

            meta_model = LogisticRegression(
                C=c_value,
                max_iter=2000,
                solver="lbfgs",
                random_state=RANDOM_STATE,
            )
            meta_model.fit(X_meta_tr, y_meta_tr)
            va_proba = meta_model.predict_proba(X_meta_va)[:, 1]
            stack_oof[va_idx] = va_proba

        thr, score = best_threshold_score(y.values, stack_oof, thresholds)
        print(f"  Stack C={c_value:.2f}: CV={score:.5f} at thr={thr:.2f}")

        if score > best_result["score"]:
            best_result = {
                "oof": stack_oof.copy(),
                "threshold": thr,
                "score": score,
                "best_c": c_value,
            }

    return (
        best_result["oof"],
        best_result["threshold"],
        best_result["score"],
        best_result["best_c"],
    )


def fit_full_models_predict_test(models, X, y, X_test):
    test_pred_map = {}

    for model_name, model in models.items():
        clf = clone(model)
        clf.fit(X, y)
        test_pred_map[model_name] = clf.predict_proba(X_test)[:, 1]

    return test_pred_map

In [18]:
model_names, best_weight_info = search_weighted_ensemble(oof_pred_map, y, THRESHOLDS)

oof_matrix = np.column_stack([oof_pred_map[name] for name in model_names])
weighted_oof = oof_matrix @ best_weight_info["weights"]

weighted_best_thr = best_weight_info["threshold"]
weighted_best_cv = best_weight_info["score"]

print("\nWeighted Ensemble")
print("Model order:", model_names)
print("Weights:", np.round(best_weight_info["weights"], 3))
print(f"Best CV: {weighted_best_cv:.5f} at threshold {weighted_best_thr:.2f}")

meta_features = oof_matrix
stack_oof, stack_best_thr, stack_best_cv, stack_best_c = stacking_oof_predictions(
    meta_features=meta_features,
    y=y,
    skf=skf,
    thresholds=THRESHOLDS,
 )

stack_meta_model = LogisticRegression(
    C=stack_best_c,
    max_iter=2000,
    solver="lbfgs",
    random_state=RANDOM_STATE,
 )
stack_meta_model.fit(meta_features, y)

print("\nStacking (LogisticRegression Meta-Model)")
print(f"Best C: {stack_best_c}")
print(f"Best CV: {stack_best_cv:.5f} at threshold {stack_best_thr:.2f}")


Weighted Ensemble
Model order: ['lgbm_main', 'lgbm_alt', 'xgboost', 'random_forest']
Weights: [0.218 0.11  0.05  0.622]
Best CV: 0.81065 at threshold 0.51
  Stack C=0.25: CV=0.80870 at thr=0.48
  Stack C=0.50: CV=0.80904 at thr=0.49
  Stack C=1.00: CV=0.80847 at thr=0.49
  Stack C=2.00: CV=0.80916 at thr=0.50
  Stack C=4.00: CV=0.80858 at thr=0.49

Stacking (LogisticRegression Meta-Model)
Best C: 2.0
Best CV: 0.80916 at threshold 0.50


In [19]:
summary_rows = []

for model_name, result in model_cv_results.items():
    summary_rows.append({
        "model": model_name,
        "cv_accuracy": result["best_cv"],
        "best_threshold": result["best_threshold"],
    })

summary_rows.append({
    "model": "weighted_ensemble",
    "cv_accuracy": weighted_best_cv,
    "best_threshold": weighted_best_thr,
})
summary_rows.append({
    "model": "stacking_logreg",
    "cv_accuracy": stack_best_cv,
    "best_threshold": stack_best_thr,
})

scoreboard = pd.DataFrame(summary_rows).sort_values("cv_accuracy", ascending=False).reset_index(drop=True)
print(scoreboard.to_string(index=False))

            model  cv_accuracy  best_threshold
weighted_ensemble     0.810652            0.51
  stacking_logreg     0.809157            0.50
          xgboost     0.808927            0.52
        lgbm_main     0.806856            0.48
    random_forest     0.806511            0.50
         lgbm_alt     0.805706            0.48


In [20]:
test_pred_map = fit_full_models_predict_test(models, X, y, X_test)

test_matrix = np.column_stack([test_pred_map[name] for name in model_names])
weighted_test_proba = test_matrix @ best_weight_info["weights"]
stack_test_proba = stack_meta_model.predict_proba(test_matrix)[:, 1]

print("Generated full-data test probabilities for all models")

Generated full-data test probabilities for all models


In [21]:
if stack_best_cv >= weighted_best_cv:
    final_name = "stacking_logreg"
    final_test_proba = stack_test_proba
    final_threshold = stack_best_thr
    final_cv_score = stack_best_cv
else:
    final_name = "weighted_ensemble"
    final_test_proba = weighted_test_proba
    final_threshold = weighted_best_thr
    final_cv_score = weighted_best_cv

final_test_pred = (final_test_proba >= final_threshold).astype(bool)

print(f"Selected final strategy: {final_name}")
print(f"Final CV score: {final_cv_score:.5f}")
print(f"Final threshold: {final_threshold:.2f}")

Selected final strategy: weighted_ensemble
Final CV score: 0.81065
Final threshold: 0.51


In [22]:
submission = pd.DataFrame({
    "PassengerId": passenger_ids.values,
    "Transported": final_test_pred,
})

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")

Saved submission.csv


In [23]:
print(submission.head())
print(f"Rows in submission: {len(submission)}")

if final_cv_score < 0.83:
    print("Note: CV < 0.83 in this run. Try expanding weight search and tuning model hyperparameters further.")

  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
Rows in submission: 4277
Note: CV < 0.83 in this run. Try expanding weight search and tuning model hyperparameters further.
